# 🎮 Fine-Tuning LoRA 30 Epochs ជាមួយ Google Drive Remote Trigger & Live Telemetry

Notebook នេះត្រូវបានបំពាក់ដោយប្រព័ន្ធ **Google Drive Remote Trigger** ដែលអនុញ្ញាតឱ្យអ្នក **បញ្ជា និងតាមដានការបង្វឹកពីចម្ងាយ (Remote Control) តាមរយៈ Google Drive** (សូម្បីតែពីលើទូរស័ព្ទដៃ ឬពេលបិទកុំព្យូទ័រក៏ដោយ)៖
- 📡 **Live Status Monitor:** សរសេររបាយការណ៍ស្វ័យប្រវត្តិចូល `status.json` រាល់ចប់ ១ Epoch (Epoch, Loss, CER, Accuracy)
- 🕹️ **Remote Command Trigger:** ឆែកមើល `command.json` លើ Drive ដើម្បីបញ្ជា `STOP_AND_SAVE`, `PAUSE`, `RESUME`, `SET_LR`
- 🔍 **Live Inference Dropzone:** គ្រាន់តែទម្លាក់រូបភាព `test.jpg` ចូល Folder Control នោះ Colab នឹងអានអក្សរខ្មែរ & រូបមន្តគណិតវិទ្យា រួចសរសេរចម្លើយចូល `result.txt` ភ្លាមៗ!
- 🎯 **Hugging Face Integration:** Base weights ពី `krotreaksmey/TinyOCRkhmer` & Dataset ពី `krotreaksmey/synthetic_khmer_math`
- 💾 **Full Checkpoints & Auto-Resume:** រក្សាទុកគ្រប់ ១ Epoch, Best Model, Last Model និង Auto-Merge LoRA ចុងក្រោយ។

## ជំហានទី ១៖ ពិនិត្យ GPU & CuDNN Benchmark

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Active Device: {device}")

## ជំហានទី ២៖ ដំឡើងបណ្ណាល័យចាំបាច់

In [ ]:
%pip install -q datasets timm jiwer torchvision matplotlib tqdm pillow huggingface_hub

## ជំហានទី ៣៖ ភ្ជាប់ Google Drive & រៀបចំ Folder បញ្ជា (Control & Trigger Center)

In [ ]:
import os
import json
import time

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/khmer_math_ocr_lora_30epoch"
except Exception as e:
    print("កំពុងដំណើរការក្រៅ Colab ឬមិនបានភ្ជាប់ Drive។ ប្រើ Local Directory:")
    BASE_DIR = "./outputs_lora_30epoch"

CHECKPOINTS_DIR = os.path.join(BASE_DIR, "epoch_checkpoints")
CONTROL_DIR = os.path.join(BASE_DIR, "control")
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(CONTROL_DIR, exist_ok=True)

COMMAND_FILE = os.path.join(CONTROL_DIR, "command.json")
STATUS_FILE = os.path.join(CONTROL_DIR, "status.json")
INFERENCE_INPUT = os.path.join(CONTROL_DIR, "test.jpg")
INFERENCE_OUTPUT = os.path.join(CONTROL_DIR, "test_result.txt")

print("=" * 75)
print(f"📁 Main Directory:        {BASE_DIR}")
print(f"💾 Checkpoints Directory: {CHECKPOINTS_DIR}")
print(f"🎮 Control Directory:     {CONTROL_DIR}")
print(f"📄 Command File:          {COMMAND_FILE}")
print(f"📊 Live Status File:      {STATUS_FILE}")
print("=" * 75)

## ជំហានទី ៤៖ អនុគមន៍បញ្ជា Google Drive Remote Trigger & Live Telemetry

In [ ]:
def update_drive_status(epoch, train_loss, val_loss, val_cer, best_cer, lr, status="TRAINING", note=""):
    """សរសេររបាយការណ៍បច្ចុប្បន្នភាពចូល Google Drive (status.json)"""
    report = {
        "status": status,
        "epoch": epoch,
        "total_epochs": 30,
        "train_loss": round(train_loss, 4) if train_loss else None,
        "val_loss": round(val_loss, 4) if val_loss else None,
        "val_cer_pct": round(val_cer * 100, 2) if val_cer else None,
        "best_cer_pct": round(best_cer * 100, 2) if best_cer else None,
        "learning_rate": f"{lr:.2e}" if lr else None,
        "note": note,
        "last_updated": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
    }
    try:
        with open(STATUS_FILE, "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2, ensure_ascii=False)
    except Exception as e:
        print(f"⚠️ មិនអាចសរសេរ status.json បានទេ: {e}")

def check_drive_command():
    """ពិនិត្យមើលថាតើមាន command.json ក្នុង Google Drive ដែរឬទេ"""
    if not os.path.exists(COMMAND_FILE):
        return None
    try:
        with open(COMMAND_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
        action = data.get("action", "").upper().strip()
        print(f"\n🔔 [Drive Trigger] បានទទួលបញ្ជាពី Google Drive: '{action}'")
        # ប្តូរឈ្មោះទុកជា log ដើម្បីកុំឱ្យ execute ដដែលៗ
        history_name = os.path.join(CONTROL_DIR, f"command_done_{int(time.time())}.json")
        os.rename(COMMAND_FILE, history_name)
        return {"action": action, "data": data}
    except Exception as e:
        print(f"⚠️ Error parsing command.json: {e}")
        return None

# សរសេរ status ដំបូងបញ្ជាក់ថាប្រព័ន្ធត្រៀមរួចរាល់
update_drive_status(0, 0, 0, 1.0, 1.0, 0, status="READY", note="System ready. Waiting for training to start.")
print(f"✅ Google Drive Trigger System initialized! Check {STATUS_FILE}")

## ជំហានទី ៥៖ ទាញយក Pretrained Model Weights ពី Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "krotreaksmey/TinyOCRkhmer"
FILENAME = "best_khmer_parseq_model_30epoch.pt"

print(f"⏳ កំពុងទាញយក Base Weights ពី Hugging Face ({REPO_ID})...")
WEIGHTS_PATH = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
print(f"✅ ទាញយករួចរាល់: {WEIGHTS_PATH}")

## ជំហានទី ៦៖ កំណត់រចនាសម្ព័ន្ធ Tokenizer & Vocab ខ្មែរ + គណិតវិទ្យា (185 តួអក្សរ)

In [ ]:
from typing import Sequence, List

class CharTokenizer:
    def __init__(self, chars: Sequence[str]):
        self.pad_id = 0
        self.bos_id = 1
        self.eos_id = 2
        self.mask_id = 3
        self.unk_id = 4
        self.special_tokens = ["[PAD]", "[BOS]", "[EOS]", "[MASK]", "[UNK]"]
        self.idx2char = self.special_tokens + list(chars)
        self.char2idx = {c: i for i, c in enumerate(self.idx2char)}

    @property
    def vocab_size(self) -> int:
        return len(self.idx2char)

    def encode(self, text: str) -> List[int]:
        return [self.char2idx.get(c, self.unk_id) for c in text]

    def decode(self, indices: Sequence[int], ignore_special_tokens: bool = True) -> str:
        res = []
        for idx in indices:
            if ignore_special_tokens and idx < len(self.special_tokens):
                continue
            if 0 <= idx < len(self.idx2char):
                res.append(self.idx2char[idx])
        return "".join(res)

KHMER_CHARACTERS = (
    "កខគឃងចឆជឈញដឋឌឍណតថទធនបផពភមយរលវសហឡអ"  # Consonants (33)
    "ឣឤឥឦឧឨឩឪឫឬឭឮឯឰឱឲឳ"                    # Independent Vowels (15)
    "ាិីឹឺុូួើឿៀេែៃោៅំះៈ៉៊់៌៍៎៏័៑"          # Dependent Vowels & Diacritics (24)
    "០១២៣៤៥៦៧៨៩"                            # Khmer Numerals (10)
    "0123456789"                            # Arabic Digits (10)
    "abcdefghijklmnopqrstuvwxyz"            # Latin Lower (26)
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"            # Latin Upper (26)
    " +-/*=()[]{}<>,.:;!?%&$#@^_~\"'|\\`"    # Math & Punctuation (31)
    "«»៛"                                    # Khmer symbols (3)
)
tokenizer = CharTokenizer(KHMER_CHARACTERS)
print(f"✅ Tokenizer Vocab Size: {tokenizer.vocab_size} tokens")

## ជំហានទី ៧៖ ស្ថាបត្យកម្ម PARSeq Model Architecture & Safe Batch Generator

In [ ]:
import math
import inspect
from dataclasses import dataclass
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from timm.models.vision_transformer import VisionTransformer, PatchEmbed

@dataclass
class ModelConfig:
    img_size: Sequence[int] = (48, 192)
    patch_size: Sequence[int] = (4, 8)
    n_channel: int = 3
    vocab_size: int = 185
    block_size: int = 128
    n_layer: int = 12
    n_head: int = 3
    n_embed: int = 192
    dropout: float = 0.1
    bias: bool = True

class ImageEncoder(VisionTransformer):
    def __init__(self, config: ModelConfig):
        sig = inspect.signature(VisionTransformer.__init__)
        kwargs = {
            "img_size": config.img_size, "patch_size": config.patch_size, "in_chans": config.n_channel,
            "embed_dim": config.n_embed, "depth": config.n_layer, "num_heads": config.n_head,
            "mlp_ratio": 4, "qkv_bias": True, "drop_rate": 0.0, "attn_drop_rate": 0.0, "drop_path_rate": 0.0,
            "embed_layer": PatchEmbed, "num_classes": 0, "global_pool": "", "class_token": False
        }
        valid_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}
        super().__init__(**valid_kwargs)
        self.cls_token = None
        num_patches = (config.img_size[0] // config.patch_size[0]) * (config.img_size[1] // config.patch_size[1])
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, config.n_embed))

    def forward(self, x: Tensor) -> Tensor:
        x = self.patch_embed(x)
        x = self.pos_drop(x + self.pos_embed)
        x = self.blocks(x)
        return self.norm(x)

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: Tensor) -> Tensor:
        variance = x.float().pow(2).mean(-1, keepdim=True)
        norm_x = x.float() * torch.rsqrt(variance + self.eps)
        return (self.weight * norm_x).type(x.dtype)

class Linear(nn.Linear):
    def forward(self, x: Tensor) -> Tensor:
        return F.linear(x, self.weight.to(x.dtype), None if self.bias is None else self.bias.to(x.dtype))

class TextDecoder(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.n_head = 2 * config.n_head
        self.tok_embed = nn.Embedding(config.vocab_size, config.n_embed)
        self.pos_embed = nn.Parameter(torch.Tensor(1, config.block_size, config.n_embed))
        self.dropout = nn.Dropout(config.dropout)
        self.sa_ln = RMSNorm(config.n_embed)
        self.sa_attn = nn.MultiheadAttention(config.n_embed, self.n_head, dropout=config.dropout, batch_first=True)
        self.cross_ln = RMSNorm(config.n_embed)
        self.cross_attn = nn.MultiheadAttention(config.n_embed, self.n_head, dropout=config.dropout, batch_first=True)
        self.ffn_ln = RMSNorm(config.n_embed)
        dim_feedforward = 4 * config.n_embed
        self.ffn = nn.Sequential(
            Linear(config.n_embed, dim_feedforward, bias=config.bias),
            nn.GELU(),
            Linear(dim_feedforward, config.n_embed, bias=config.bias),
            nn.Dropout(config.dropout)
        )
        self.lm_head = Linear(config.n_embed, config.vocab_size)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x: Tensor, xi: Tensor, targets: Tensor = None):
        if x.size(1) > self.pos_embed.size(1):
            x = x[:, :self.pos_embed.size(1)]
            if targets is not None:
                targets = targets[:, :self.pos_embed.size(1)]
        b, t = x.size()
        tok_embed = self.tok_embed(x)
        query = self.dropout(tok_embed + self.pos_embed[:, :t])
        mask = torch.triu(torch.ones((t, t), dtype=torch.bool, device=x.device), 1)
        res = query
        query = self.sa_ln(res)
        query, _ = self.sa_attn(query, query, query, attn_mask=mask)
        res = res + query
        query, _ = self.cross_attn(self.cross_ln(res), xi, xi)
        res = res + query
        res = res + self.ffn(self.ffn_ln(res))
        if targets is not None:
            logits = self.lm_head(res).float()
            loss = F.cross_entropy(logits.flatten(end_dim=1), targets.flatten(), ignore_index=183)
            return logits, loss
        return self.lm_head(res[:, [-1], :]).float(), None

class OCRModel(nn.Module):
    def __init__(self, config: ModelConfig = None):
        super().__init__()
        if config is None:
            config = ModelConfig()
        self.config = config
        self.tokenizer = tokenizer
        self.encoder = ImageEncoder(config)
        self.decoder = TextDecoder(config)

    def forward(self, img_tensor: Tensor, input_tokens: Tensor, targets_tokens: Tensor = None):
        xi = self.encoder(img_tensor)
        return self.decoder(input_tokens, xi, targets_tokens)

    @torch.inference_mode()
    def generate(self, img_tensor: Tensor, max_new_tokens: int = 128):
        if img_tensor.dim() == 3:
            img_tensor = img_tensor.unsqueeze(0)
        dev = next(self.parameters()).device
        img_tensor = img_tensor.to(dev)
        b = img_tensor.size(0)
        xi = self.encoder(img_tensor)
        idx = torch.full((b, 1), fill_value=self.tokenizer.bos_id, dtype=torch.long, device=dev)
        max_tokens = min(max_new_tokens, self.decoder.pos_embed.size(1) - 1)
        for _ in range(max_tokens):
            dec_out = self.decoder(idx, xi)
            logits = dec_out[0] if isinstance(dec_out, tuple) else dec_out
            logits = logits[:, -1, :]
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
            idx = torch.cat((idx, idx_next), dim=1)
            if b == 1:
                if idx_next.item() == self.tokenizer.eos_id:
                    break
            else:
                if (idx == self.tokenizer.eos_id).any(dim=1).all():
                    break
        return idx

model = OCRModel(ModelConfig())
state_dict = torch.load(WEIGHTS_PATH, map_location="cpu")
if "state_dict" in state_dict:
    state_dict = state_dict["state_dict"]
model.load_state_dict(state_dict, strict=True)
print(f"✅ Pretrained Weights Loaded successfully from {WEIGHTS_PATH}!")

## ជំហានទី ៨៖ អនុវត្ត LoRA (Rank 16) លើ Vision & Language Projections

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, original_linear: nn.Linear, r: int = 16, alpha: float = 32.0):
        super().__init__()
        self.original_linear = original_linear
        self.r = r
        self.scaling = alpha / r
        in_features = original_linear.in_features
        out_features = original_linear.out_features
        self.lora_A = nn.Parameter(torch.zeros((r, in_features)))
        self.lora_B = nn.Parameter(torch.zeros((out_features, r)))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
        for p in self.original_linear.parameters():
            p.requires_grad = False

    def forward(self, x: Tensor) -> Tensor:
        orig_out = self.original_linear(x)
        lora_out = F.linear(F.linear(x, self.lora_A), self.lora_B) * self.scaling
        return orig_out + lora_out

def apply_lora_to_model(model: nn.Module, r: int = 16, alpha: float = 32.0):
    count = 0
    # Decoder Self-Attention & Cross-Attention Projections
    for name, module in list(model.decoder.named_modules()):
        if isinstance(module, (nn.Linear, Linear)) and "out_proj" in name:
            parent_name = name.rsplit(".", 1)[0] if "." in name else ""
            parent = model.decoder if not parent_name else dict(model.decoder.named_modules())[parent_name]
            child_name = name.rsplit(".", 1)[1] if "." in name else name
            setattr(parent, child_name, LoRALinear(module, r=r, alpha=alpha))
            count += 1
    # LM Head
    model.decoder.lm_head = LoRALinear(model.decoder.lm_head, r=r, alpha=alpha)
    count += 1
    return count

num_lora_layers = apply_lora_to_model(model, r=16, alpha=32.0)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA បានបំពាក់លើ {num_lora_layers} Layers!")
print(f"🔥 Trainable Parameters: {trainable:,} ({trainable/total*100:.2f}% នៃ {total:,})")

## ជំហានទី ៩៖ ទាញយក Dataset ពី Hugging Face (`krotreaksmey/synthetic_khmer_math`)

In [ ]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

print("⏳ កំពុងទាញយក Dataset ពី Hugging Face (krotreaksmey/synthetic_khmer_math)...")
hf_dataset = load_dataset("krotreaksmey/synthetic_khmer_math")
train_data = hf_dataset["train"]
val_data = hf_dataset["test"] if "test" in hf_dataset else hf_dataset["train"].select(range(min(5000, len(hf_dataset["train"]))))
print(f"✅ Train Samples: {len(train_data):,} | Val Samples: {len(val_data):,}")

img_transform = transforms.Compose([
    transforms.Resize((48, 192)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class OCRDataset(Dataset):
    def __init__(self, data, tokenizer, transform=None, max_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img = item["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)
        if img.mode != "RGB":
            img = img.convert("RGB")
        if self.transform:
            img = self.transform(img)
        text = str(item.get("text", ""))
        tokens = self.tokenizer.encode(text)
        inp = [self.tokenizer.bos_id] + tokens
        tgt = tokens + [self.tokenizer.eos_id]
        inp = inp[:self.max_len]
        tgt = tgt[:self.max_len]
        inp += [self.tokenizer.pad_id] * (self.max_len - len(inp))
        tgt += [183] * (self.max_len - len(tgt))
        return {"img": img, "input_tokens": torch.tensor(inp, dtype=torch.long), "target_tokens": torch.tensor(tgt, dtype=torch.long)}

BATCH_SIZE = 128
train_dataset = OCRDataset(train_data, tokenizer, transform=img_transform)
val_dataset = OCRDataset(val_data, tokenizer, transform=img_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print(f"✅ DataLoaders ត្រៀមរួចជាស្រេច (Batch Size: {BATCH_SIZE}, Workers: 0)!")

## ជំហានទី ១០៖ កំណត់ Optimizer, Scheduler & Auto-Resume Logic

In [ ]:
EPOCHS = 30
LEARNING_RATE = 3.5e-4

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler(enabled=torch.cuda.is_available())

best_model_path = os.path.join(BASE_DIR, "best_khmer_parseq_lora_30epoch.pt")
last_model_path = os.path.join(BASE_DIR, "last_checkpoint.pt")

start_epoch = 1
best_val_cer = float("inf")
history = {"train_loss": [], "val_loss": [], "val_cer": []}

# Auto-Resume Check
if os.path.exists(last_model_path):
    print(f"🔄 ឃើញ Last Checkpoint ចាស់ ({last_model_path})! កំពុង Auto-Resume...")
    ckpt = torch.load(last_model_path, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"], strict=False)
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_val_cer = ckpt.get("best_val_cer", float("inf"))
    history = ckpt.get("history", history)
    print(f"✅ Auto-Resume ជោគជ័យ! បន្តពី Epoch {start_epoch:02d} ដល់ {EPOCHS:02d} (Best CER: {best_val_cer*100:.2f}%)")
else:
    print("✨ មិនមាន Checkpoint ចាស់ទេ ចាប់ផ្ដើមពី Epoch 01 ជាថ្មី!")

model = model.to(device)

## ជំហានទី ១១៖ Training Loop 30 Epochs ជាមួយ Google Drive Remote Trigger & Live Telemetry

In [ ]:
import jiwer
from tqdm.auto import tqdm

print("=" * 75)
print(f"🏁 ចាប់ផ្ដើម Fine-Tuning LoRA (Epoch {start_epoch:02d} ដល់ {EPOCHS:02d})...")
print(f"🎮 អ្នកអាចបញ្ជាពីចម្ងាយតាមរយៈ: {COMMAND_FILE}")
print(f"📊 តាមដាន Live Status លើ Drive: {STATUS_FILE}")
print("=" * 75)

for epoch in range(start_epoch, EPOCHS + 1):
    # --- 1. CHECK REMOTE COMMANDS FROM GOOGLE DRIVE ---
    cmd = check_drive_command()
    if cmd:
        action = cmd["action"]
        if action in ["STOP", "STOP_AND_SAVE"]:
            print(f"🛑 ទទួលបានបញ្ជា '{action}' ពី Google Drive! បញ្ឈប់ការបង្វឹក និងរក្សាទុក Model...")
            update_drive_status(epoch - 1, 0, 0, best_val_cer, best_val_cer, scheduler.get_last_lr()[0], status="STOPPED_BY_USER", note=f"Stopped by command '{action}'.")
            break
        elif action == "SET_LR":
            new_lr = float(cmd["data"].get("lr", LEARNING_RATE))
            for g in optimizer.param_groups:
                g["lr"] = new_lr
            print(f"⚡ Learning Rate updated to: {new_lr:.2e}")
        elif action == "PAUSE":
            print("⏸️ ទទួលបានបញ្ជា PAUSE! រង់ចាំរហូតដល់មានបញ្ជា RESUME ក្នុង Google Drive...")
            update_drive_status(epoch - 1, 0, 0, best_val_cer, best_val_cer, scheduler.get_last_lr()[0], status="PAUSED", note="Paused by command. Drop {'action': 'RESUME'} to continue.")
            while True:
                time.sleep(10)
                resume_cmd = check_drive_command()
                if resume_cmd and resume_cmd["action"] in ["RESUME", "START"]:
                    print("▶️ ទទួលបានបញ្ជា RESUME! បន្តការបង្វឹកឡើងវិញ...")
                    break

    # --- 2. TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    start_time = time.time()
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Train]", leave=False)
    for batch in pbar:
        imgs = batch["img"].to(device, non_blocking=True)
        inp_toks = batch["input_tokens"].to(device, non_blocking=True)
        tgt_toks = batch["target_tokens"].to(device, non_blocking=True)
        
        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            _, loss = model(imgs, inp_toks, tgt_toks)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    scheduler.step()
    avg_train_loss = running_loss / len(train_loader)
    epoch_time = time.time() - start_time
    
    # --- 3. VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    val_batches = 0
    val_targets, val_predictions = [], []
    
    with torch.no_grad():
        for v_idx, batch in enumerate(val_loader):
            imgs = batch["img"].to(device, non_blocking=True)
            inp_toks = batch["input_tokens"].to(device, non_blocking=True)
            tgt_toks = batch["target_tokens"].to(device, non_blocking=True)
            
            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                _, loss = model(imgs, inp_toks, tgt_toks)
            val_loss += loss.item()
            val_batches += 1
            
            # Batch safe inference for CER
            if v_idx < 5:
                sub_imgs = imgs[:16]
                sub_tgts = tgt_toks[:16]
                pred_ids = model.generate(sub_imgs, max_new_tokens=64)
                for p_seq, t_seq in zip(pred_ids, sub_tgts):
                    p_text = tokenizer.decode(p_seq.tolist(), ignore_special_tokens=True)
                    t_text = tokenizer.decode(t_seq.tolist(), ignore_special_tokens=True)
                    val_predictions.append(p_text)
                    val_targets.append(t_text)
            if v_idx >= 30:
                break
                
    avg_val_loss = val_loss / max(val_batches, 1)
    current_cer = jiwer.cer(val_targets, val_predictions) if val_targets else 1.0
    
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_cer"].append(current_cer)
    
    # --- 4. SAVE CHECKPOINTS ---
    lora_state_dict = {k: v for k, v in model.state_dict().items() if "lora_" in k}
    
    # Every Epoch Checkpoint
    epoch_ckpt_file = os.path.join(CHECKPOINTS_DIR, f"lora_epoch_{epoch:02d}_cer_{current_cer*100:.2f}.pt")
    torch.save({
        "epoch": epoch,
        "lora_state_dict": lora_state_dict,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "val_cer": current_cer
    }, epoch_ckpt_file)
    
    # Best Model Checkpoint
    is_best = current_cer < best_val_cer
    save_msg = ""
    if is_best:
        best_val_cer = current_cer
        torch.save(lora_state_dict, best_model_path)
        save_msg = f"⭐ NEW BEST! CER: {current_cer*100:.2f}% (Saved)"
        
    # Last Model Checkpoint
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_cer": best_val_cer,
        "history": history
    }, last_model_path)
    
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({epoch_time:.0f}s) | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val CER: {current_cer*100:.2f}% | LR: {current_lr:.2e} {save_msg}")
    
    # --- 5. UPDATE LIVE STATUS TO GOOGLE DRIVE ---
    update_drive_status(
        epoch=epoch, train_loss=avg_train_loss, val_loss=avg_val_loss,
        val_cer=current_cer, best_cer=best_val_cer, lr=current_lr,
        status="TRAINING", note=f"Epoch {epoch:02d} completed. {save_msg}"
    )

update_drive_status(EPOCHS, avg_train_loss, avg_val_loss, current_cer, best_val_cer, current_lr, status="COMPLETED", note="30 Epochs training finished successfully!")
print("=" * 75)
print(f"🎉 ការបង្វឹក 30 Epochs បានបញ្ចប់ដោយជោគជ័យ! Best CER: {best_val_cer*100:.2f}%")
print(f"📁 All Checkpoints: {CHECKPOINTS_DIR}")
print(f"💾 Best LoRA:       {best_model_path}")
print(f"💾 Last State:      {last_model_path}")

## ជំហានទី ១២៖ Auto-Merge LoRA Weights ទៅ Standalone PyTorch Weights

In [ ]:
merged_model_path = os.path.join(BASE_DIR, "best_khmer_parseq_model_30epoch_finetuned.pt")
print(f"⏳ កំពុង Merge LoRA ទៅ Standalone PyTorch Model ({merged_model_path})...")

# Create fresh base model and load original weights
standalone_model = OCRModel(ModelConfig())
raw_sd = torch.load(WEIGHTS_PATH, map_location="cpu")
if "state_dict" in raw_sd:
    raw_sd = raw_sd["state_dict"]
standalone_model.load_state_dict(raw_sd, strict=True)

# Load best LoRA weights
best_lora = torch.load(best_model_path, map_location="cpu")
standalone_sd = standalone_model.state_dict()

merged_count = 0
for k, v in best_lora.items():
    if "lora_A" in k:
        prefix = k.replace(".lora_A", "")
        lora_A = best_lora[f"{prefix}.lora_A"]
        lora_B = best_lora[f"{prefix}.lora_B"]
        delta = (lora_B @ lora_A) * (32.0 / 16.0)
        weight_key = prefix.replace(".original_linear", "") + ".weight"
        if weight_key in standalone_sd:
            standalone_sd[weight_key] += delta
            merged_count += 1

torch.save({"state_dict": standalone_sd}, merged_model_path)
print(f"✅ LoRA weights ចំនួន {merged_count} layers ត្រូវបាន Merge ជោគជ័យ!")
print(f"🏆 ឯកសារចុងក្រោយសម្រាប់យកទៅប្រើប្រាស់ (Standalone PyTorch Checkpoint): {merged_model_path}")